# Chapter 4 · Networks to Predict Text

We will follow the question **start, start, a → n** from the name ana. First we do a hand calculation with values chosen to make it easy to understand. Then we run the real model, with its own initialization and training. Do not confuse the illustrative numbers with the results of learning.

The notebook requires PyTorch on the CPU. Google Colab already has it installed; on your computer, install it with `pip install torch`. Nothing is installed when these cells run.

**Training:** ana, anna, alba and alan; 19 questions to fit the parameters.

**Held-out set:** lana and bala, as in chapters 2 and 3; ten questions to check the result at the end. We do not fit any parameter with them, nor do we choose another configuration based on their loss.

## From three cards to a probability

Start is represented by [0.1, −0.2] and a by [0.7, 0.3]. When we join the three cards of start, start, a, we get six components. These values are illustrative.

In the first unit we choose a weight of 2 for the first component and zero for the others; the bias is 0.1. We apply tanh. In the final layer, output n receives twice that activation and the other four receive zero.

The following code does those same calculations. `x_manual[0]` selects the first component, `math.tanh` applies the curve and `math.exp` computes the exponential. The denominator adds the four exponentials equal to one and the one for n. The cost is the negative natural logarithm of the probability of n.

In [1]:
import math
x_manual = [0.1, -0.2, 0.1, -0.2, 0.7, 0.3]
u_manual = 2 * x_manual[0] + 0.1
h_manual = math.tanh(u_manual)
z_n_manual = 2 * h_manual
p_n_manual = math.exp(z_n_manual) / (4 + math.exp(z_n_manual))
cost_manual = -math.log(p_n_manual)
print("Sum:", u_manual)
print("Activation:", h_manual)
print("Logit of n:", z_n_manual)
print("Probability of n:", p_n_manual)
print("Cost:", cost_manual)

Sum: 0.30000000000000004
Activation: 0.29131261245159096
Logit of n: 0.5826252249031819
Probability of n: 0.3092412026392393
Cost: 1.1736337155216467


## What changes when we train

The following program keeps the organization we have just walked through, but **it does not use the hand-picked weights**. It initializes its 111 parameters with seed 42 and corrects them with the 19 questions from the four names. The held-out set is not added to the training.

## Preparing the names and the two lists of symbols

This block prepares the data and the names that we will use in the rest of the program. It does not make any prediction yet.

**The tools.** The first line describes the file. `math` lets us compute the logarithm of the uniform baseline; `time` will be used to measure the duration. `torch` is PyTorch, the library that handles tensors and derivatives. From it we import `nn`, which contains the layers, and `functional`, which we give the short name `F`. That last module contains the loss function.

**The data.** The two tuples separate the training set and the held-out set. `INPUTS` lists the five symbols allowed in a context; `OUTPUTS` lists the five possible answers. These strings are vocabularies, not examples that are given to the network. Their order assigns the identifiers.

`enumerate(INPUTS)` goes through positions and symbols: first position 0 with start, then 1 with a, and so on. The expression in braces stores the correspondence in a dictionary. So looking up `TO_INPUT['a']` returns 1. The second dictionary works the same way with the list of outputs; looking up n returns 3.

The last two lines set the size of the network and of the training. The context has three positions; each card, two components; the intermediate layer, eight units (`HIDDEN`). There will be 400 updates with rate 0.1 and an initialization with seed 42. These constants describe the real experiment, not the weights chosen by hand to explain the arithmetic.

In [2]:
"""Chapter 4: character MLP. PyTorch on CPU, with no data downloads."""
import math
import time
import torch
from torch import nn
from torch.nn import functional as F

TRAINING = ("ana", "anna", "alba", "alan")
HELD_OUT = ("lana", "bala")
INPUTS = "^abln"
OUTPUTS = "abln$"
TO_INPUT = {c: i for i, c in enumerate(INPUTS)}
TO_OUTPUT = {c: i for i, c in enumerate(OUTPUTS)}
CONTEXT, DIMENSION, HIDDEN = 3, 2, 8
STEPS, RATE, SEED = 400, 0.1, 42

## Turning a name into training rows

The function receives names and returns two collections: the numerical contexts and their targets. With ana it should produce the four rows we followed on paper.

`xs` and `ys` start out as empty lists. In the first we will store inputs; in the second, answers. The outer loop takes one name at a time. The condition rejects an empty string or any character that is not among the four allowed letters. `OUTPUTS[:-1]` means “all the outputs except the last one,” which leaves out the end.

The window starts as [0,0,0]. Multiplying a list with a single zero by three repeats that element. The inner loop goes through the letters of the name and an end mark. In each round, `append` stores one row in each list. `copy()` stores a copy of the window, and the output dictionary converts the answer into its number.

**Let us follow ana.** The first round stores context [0,0,0] and target 0, which means a. It then updates the window to [0,0,1]. In the second round it stores that window and target 3, which means n. This is exactly our central question. After that, it continues with the last a and with the end.

`context[1:]` removes the oldest slot. Adding the two lists puts the new card on the right. We do not make that update when the output is the end, because there will be no further prediction within that name.

The check `if not xs` avoids returning an empty set. The last lines convert the lists into integer CPU tensors. The function returns both together: that is why we can write `x, y = prepare(...)`. With the four names, x has 19 rows of three numbers, and y has 19 answers.

In [3]:
def prepare(names):
    xs, ys = [], []
    for name in names:
        if not name or any(c not in OUTPUTS[:-1] for c in name):
            raise ValueError("Empty name or unknown letter")
        context = [TO_INPUT["^"]] * CONTEXT
        for next_char in name + "$":
            xs.append(context.copy())
            ys.append(TO_OUTPUT[next_char])
            if next_char != "$":
                context = context[1:] + [TO_INPUT[next_char]]
    if not xs:
        raise ValueError("Empty collection")
    return (torch.tensor(xs, dtype=torch.long, device="cpu"),
            torch.tensor(ys, dtype=torch.long, device="cpu"))

## Building the path that a prediction follows

This function creates the network and returns it. Each line adds one operation to the path a context follows. The parameters are initialized here; they have not yet been corrected with the names.

`nn.Sequential` means that the operations are run one after another. When we later call `model(x)`, PyTorch passes the output of each line on to the next.

**Looking up cards.** The first line creates `Embedding(5, 2)`: five cards, two numbers per card. With an input [0,0,1], it will look up the start card twice and the card for a once. Those contents will be learned during training.

**Laying out the six numbers.** `Flatten(start_dim=1)` keeps the dimension of the examples and flattens the dimensions that come after it. For a single question it turns a table of three pairs into a row of six components. It does not mix the questions with one another.

**Combining and transforming.** The first `Linear` layer receives six numbers and produces eight weighted sums, with their biases. `Tanh` transforms each one into an activation. The last layer receives eight activations and produces five logits, one per possible output. Linear layers include biases by default.

`.to("cpu")` places the parameters on the CPU. The result of this function is the model with its 111 adjustable numbers. We do not include softmax at the end because, during training, the loss function will receive the logits directly. The conversion into probabilities will appear when we generate a letter.

In [4]:
def create_model():
    return nn.Sequential(
        nn.Embedding(len(INPUTS), DIMENSION),
        nn.Flatten(start_dim=1),
        nn.Linear(CONTEXT * DIMENSION, HIDDEN),
        nn.Tanh(),
        nn.Linear(HIDDEN, len(OUTPUTS)),
    ).to("cpu")

## Repeating the prediction and the correction

There are two different functions here. `evaluate` returns a loss value without changing the model. `train` uses the loss to modify the parameters and returns how the loss evolved.

In `evaluate`, `model(x)` computes one row of logits per context. The loss compares each row with its target in y and averages the costs. `.item()` turns that single-value result into a Python number. `no_grad()` indicates that we will not need derivatives in this evaluation.

In `train` we create the optimizer. `model.parameters()` provides the cards, weights and biases that it can modify. `lr=RATE` sets the size of the corrections. The history first stores the loss of the initial model, associated with step zero.

**One round of the loop.** First, the previous gradients are discarded with `zero_grad`. Then the logits and the current loss are computed. `backward()` computes the derivatives. `step()` uses them to update the parameters. The next round will predict again with the new values.

`range(1, STEPS + 1)` goes from step 1 to step 400, both included. Every step uses the 19 training questions together. The condition on the remainder of dividing by 50 picks out steps 50, 100 and the other multiples. At those steps the loss is measured again and stored in the history.

The held-out set does not appear in this function. Nor do we stop training when a generated sample looks nice. We finish at the fixed number of steps. These layers do not behave differently in training and in evaluation; `no_grad` only avoids preparing derivatives when they are not needed.

In [5]:
def evaluate(model, x, y):
    with torch.no_grad():
        return F.cross_entropy(model(x), y).item()


def train(model, x, y):
    optimizer = torch.optim.SGD(model.parameters(), lr=RATE)
    history = [(0, evaluate(model, x, y))]
    for step in range(1, STEPS + 1):
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            history.append((step, evaluate(model, x, y)))
    return history

## Computing the baseline from chapter 2

These functions rebuild and evaluate the bigram. They do it with tensors, but the arithmetic is the same as in chapter 2: count continuations and normalize each row.

`torch.ones` creates a five-by-five table filled with ones. Each one represents the slip added by smoothing. We have not counted any name yet.

`zip(x, y)` takes each input together with its target. Of the input, it only uses `row[-1]`, its last slot. For our question [0,0,1] with target 3, it increases the cell in row 1 and column 3: after a, n was observed. The two oldest positions play no part, because this is still a bigram.

The sum with `dim=1` gives one total per row. `keepdim=True` keeps each total as a column, so that the five cells of that row are divided by the same number. For example, the row for a adds up to thirteen after smoothing. The division produces its probability distribution.

In the evaluation, `x[:, -1]` selects the last slot of every question. The second index, y, selects the real output of each one. In this way we get one probability per transition, not a new prediction chosen at random.

`log()` computes the natural logarithm of each probability. `mean()` takes the mean, the minus sign gives the cost and `item()` returns the number. It is the same procedure we already knew. Smoothing avoids zero probabilities in this table.

In [6]:
def fit_bigrams(x, y):
    counts = torch.ones(len(INPUTS), len(OUTPUTS))
    for row, target in zip(x, y):
        counts[row[-1], target] += 1
    return counts / counts.sum(dim=1, keepdim=True)


def evaluate_bigrams(table, x, y):
    probabilities = table[x[:, -1], y]
    return -probabilities.log().mean().item()

## Writing a name with the learned model

The function receives a trained model and a random generator. It returns the written text and a flag that distinguishes ending normally from reaching the step limit.

We start with three start marks and an empty list of letters. We are not learning, so the loop works inside `no_grad()`. The limit allows up to twenty decisions with the default value.

`[context]` adds the brackets of a list of questions: there is a single question with three identifiers. Converted into a tensor, it has shape (1,3). The model returns five logits in one row. `softmax(dim=1)` turns them into a distribution, and `[0]` extracts that single row.

`torch.multinomial` selects an index at random according to those probabilities; the argument 1 asks for a single choice. `generator=rng` says which random generator to use, and `item()` extracts the index from the result.

**If index 0 comes out**, `OUTPUTS[0]` translates it into the letter a. We add it to the text and look up its input card, which is 1, to shift the window. The new context will be [0,0,1]. This translation between the two lists is necessary: copying the selected index directly would give a wrong context.

If the end appears, we join the letters with `join` and return `True`. The end symbol is not added. If the steps run out, we return `False`. The text can be empty if the first random selection already chooses the end. We do not draw again at random to hide it.

In [7]:
def generate(model, rng, max_steps=20):
    context = [TO_INPUT["^"]] * CONTEXT
    letters = []
    with torch.no_grad():
        for _ in range(max_steps):
            x = torch.tensor([context], dtype=torch.long)
            probabilities = model(x).softmax(dim=1)[0]
            index = torch.multinomial(
                probabilities, 1, generator=rng).item()
            next_char = OUTPUTS[index]
            if next_char == "$":
                return "".join(letters), True
            letters.append(next_char)
            context = context[1:] + [TO_INPUT[next_char]]
    return "".join(letters), False

## Running and reading the experiment

The `run` function organizes the previous ones. First it prepares the data, trains and gathers the measurements.

We limit PyTorch to one CPU thread because the job is very small. We fix the seed before creating the network, so that the initialization can be repeated. The calls to `prepare` build the training questions and the held-out questions separately.

`create_model()` returns the initial network. The call to `train` receives only x and y: the held-out data, xh and yh, do not go into it. The variable `history` receives the pairs of step and loss that were saved during learning.

The clock is read before and after training. The difference measures that whole call, including the optimizer and the periodic evaluations. It does not include installing PyTorch, importing it or generating the final names. That is why this time may be shorter than what you notice when you start the whole file.

Then the bigram is fitted with x and y. The dictionary `results` stores a name and a value for each measurement: uniform, bigrams on both sets and MLP on both sets. The evaluation functions only query the models.

The parameter count adds up `numel()`, the number of values in each adjustable tensor. It should give 111. The last entry of the dictionary stores the measured seconds. So far we have gathered the measurements; we still have to print them and generate samples.

The first prints show the PyTorch version, the shapes of x and y, and the history. It is useful to check the shapes before reading the loss: there should be 19 questions of three identifiers and 19 targets.

`results.items()` lets us go through the names of the measurements together with their values. Then we create the sample generator on the CPU with seed 7. This seed controls the random selections and is independent of the one used to create the weights. The generator is prepared outside the loop so that it moves forward from one sample to the next.

In each of the ten rounds, `generate` returns a text and a flag. `repr(text)` prints the text in quotes, which makes an empty string visible. The conditional expression chooses the word “end” or “truncated.” It does not print only the samples that look like known names.

The function also returns the model, the history and the measurements, so that they can be inspected in a notebook. In the program file, a final condition on `__name__` calls `run()` if we start the file as a program; here we call it in the last line of the cell.

In the book, this function is shown in two blocks. In the notebook they are combined in a single cell, so that Python receives the complete definition before running it.

In [8]:
def run():
    torch.set_num_threads(1)
    torch.manual_seed(SEED)
    x, y = prepare(TRAINING)
    xh, yh = prepare(HELD_OUT)
    model = create_model()
    start = time.perf_counter()
    history = train(model, x, y)
    seconds = time.perf_counter() - start
    bigrams = fit_bigrams(x, y)
    results = {
        "uniform": math.log(len(OUTPUTS)),
        "bigrams_training": evaluate_bigrams(bigrams, x, y),
        "bigrams_held_out": evaluate_bigrams(bigrams, xh, yh),
        "mlp_training": evaluate(model, x, y),
        "mlp_held_out": evaluate(model, xh, yh),
        "parameters": sum(p.numel() for p in model.parameters()),
        "training_seconds": seconds,
    }
    print("PyTorch:", torch.__version__, "Device: CPU")
    print("Shapes:", tuple(x.shape), tuple(y.shape))
    print("History:", history)
    for name, value in results.items():
        print(name, value)
    rng = torch.Generator(device="cpu").manual_seed(7)
    for _ in range(10):
        text, complete = generate(model, rng)
        print(repr(text), "end" if complete else "truncated")
    return model, history, results

model, history, results = run()

PyTorch: 2.8.0+cpu Device: CPU
Shapes: (19, 3) (19,)
History: [(0, 1.5178232192993164), (50, 1.2839688062667847), (100, 1.0052188634872437), (150, 0.707410991191864), (200, 0.544045627117157), (250, 0.4629152715206146), (300, 0.4168529808521271), (350, 0.38598737120628357), (400, 0.3645842671394348)]
uniform 1.6094379124341003
bigrams_training 1.1139613389968872
bigrams_held_out 1.4099500179290771
mlp_training 0.3645842671394348
mlp_held_out 2.043356418609619
parameters 111
training_seconds 2.200626000063494
'alan' end
'alan' end
'anna' end
'a' end
'alan' end
'alna' end
'anna' end
'alba' end
'alba' end
'alan' end


## Looking at the question we have followed

We prepare only ana. The second row, index 1 because counting starts at zero, should be [0,0,1], and its target should be 3. We convert the tensors into lists or numbers with `tolist` and `item` so that we can read them. The loop goes through the four questions of that name; it does not train again.

In [9]:
x_ana, y_ana = prepare(("ana",))
for context, target in zip(x_ana, y_ana):
    print(context.tolist(), "target:", target.item())

[0, 0, 0] target: 0
[0, 0, 1] target: 3
[0, 1, 4] target: 0
[1, 4, 1] target: 4


## Observing the dimensions of the program

We use the 19 training questions. Each layer receives that same group, transformed. The loop prints the name of the layer and the shape of its result. `no_grad` avoids preparing derivatives for this observation. The first size is still 19. When the cards are looked up, the data go from two dimensions, (19,3), to three, (19,3,2).

In [10]:
x, y = prepare(TRAINING)
v = x
with torch.no_grad():
    for layer in model:
        v = layer(v)
        print(type(layer).__name__, tuple(v.shape))

Embedding (19, 3, 2)
Flatten (19, 6)
Linear (19, 8)
Tanh (19, 8)
Linear (19, 5)


## A derivative you can check by hand

This example is independent of the network for names. `requires_grad=True` says that we want the derivative with respect to w. We compute u=2w−3 and L=u². `backward` computes the derivative and `w.grad.item()` shows the result, which should be −4. The value of w has not been changed yet.

In [11]:
w = torch.tensor(1.0, requires_grad=True)
u = 2 * w - 3
loss = u ** 2
loss.backward()
print("Gradient:", w.grad.item())

Gradient: -4.0


## Exercises

1. Write down the five contexts and targets that `anna` produces. Keep the input and the expected answer separate.

2. Translate the context start, start, a into input identifiers. Translate the target n into its output identifier. Explain the role of each number.

3. Distinguish between the identifier of a letter and its embedding. Say which one stays fixed and which one changes during training.

4. Retrieve the three illustrative cards for start, start, a and flatten them. Point out which two components correspond to the most recent letter.

5. Rebuild the 111 parameters without looking at the total: count the cards, weights and biases of each layer.

6. Follow the first unit of the hand-worked example: weights [2,0,0,0,0,0], bias 0.1 and the illustrative input. Compute its sum and its tanh activation.

7. Follow output n of the hand-worked example: its only nonzero weight is 2, and it receives the previous activation. Compute its logit, its probability when the other four logits are zero and the cost if n is the target.

8. Explain what `F.cross_entropy` receives. Distinguish computing the loss from sampling the next letter.

9. For u=2w-3 and L=u^2, compute the derivative at w=1 and one update with rate 0.1. Relate the calculation to `backward()` and `step()`.

10. Describe the shapes `(19, 3)`, `(19, 3, 2)`, `(19, 5)` and `(19,)`. Say how many dimensions each tensor has and what each size means.

11. Read the table of real results: say where the network improves on the bigram and where it does worse. Explain why the two results do not contradict each other.

12. Suppose the random selection returns output identifier 0. Write the generated letter and the identifier we add to the context. Also explain what happens if the end comes out or the limit runs out.

## My conclusion

Explain in your own words the path from start, start, a to the loss. Then distinguish the hand-worked example from the real training, and compare the two columns of results.

My answer:

## References

The teaching path is inspired by [makemore](https://github.com/karpathy/makemore), by Andrej Karpathy. The layers and the loss are implemented with PyTorch 2.8.0. The references and the full explanations are in chapter 4 of the book.